# Synthetic Data Generation for RAG Evaluation

Session 1 built a vector RAG application over a cat health guideline PDF. This
session creates an evaluation dataset for that application and uses the dataset
to compare two retrieval configurations. All generation, embedding, RAG, and
judge requests are routed through Vercel AI Gateway.

The workflow is:

~~~text
corpus -> knowledge graph -> synthetic examples -> human review
       -> LangSmith dataset -> baseline and candidate experiments
~~~

Synthetic examples reduce the cost of getting started, but generated references
are not automatically ground truth. We will inspect and curate them before using
them as evaluation targets.

> This is an educational cat health exercise, not veterinary advice. Generated
> questions and answers must not be used to diagnose, prescribe, or replace a
> veterinarian.

## Learning Outcomes

By the end of this notebook, you will be able to:

- Explain how Ragas builds a knowledge graph for test data generation.
- Distinguish single-hop specific, multi-hop specific, and multi-hop abstract queries.
- Generate and review synthetic questions, reference contexts, and reference answers.
- Route generation, embeddings, RAG, and judge calls through Vercel AI Gateway.
- Upload reviewed examples to a LangSmith dataset.
- Evaluate answer correctness, answer groundedness, and retrieval relevance.
- Run a controlled RAG experiment that changes one variable at a time.

## Table of Contents

- **Breakout Room #1: Synthetic Test Data with Ragas**
  - Task 1: Environment Setup
  - Task 2: Load the Cat Health Corpus
  - Task 3: Build and Enrich a Knowledge Graph
  - Task 4: Inspect the Query Distribution
  - Task 5: Generate and Inspect a Synthetic Test Set
  - Activity #1: Review and Curate the Dataset
- **Breakout Room #2: RAG Evaluation with LangSmith**
  - Task 6: Create a LangSmith Dataset
  - Task 7: Build a Baseline RAG Application
  - Task 8: Define RAG Evaluators
  - Task 9: Run the Baseline Experiment
  - Task 10: Change One Retrieval Variable and Re-Evaluate
  - Activity #2: Compare, Diagnose, and Iterate
  - Advanced Build: Add Robustness and Adversarial Cases

---
# Breakout Room #1
## Synthetic Test Data with Ragas

Ragas uses the source corpus to create a richer representation of its topics and
relationships. Query synthesizers then select scenarios from that representation
and generate questions plus reference answers.

The knowledge graph is a generation aid. It is not the graph used by the RAG
application in Breakout Room #2.

## Task 1: Environment Setup

From the <code>05_Synthetic_Data_Generation_for_RAG_Evals</code> folder:

~~~bash
uv sync
~~~

Then select the environment created by uv as this notebook's kernel.

Required accounts:

- Vercel AI Gateway for generation, embeddings, the RAG answer model, and judges
- LangSmith for the dataset and experiments

A direct OpenAI API key is not required. The OpenAI SDK is used only as a
protocol-compatible client pointed at Vercel AI Gateway.

The default synthetic test set is intentionally small. Ragas generation and
LLM-as-judge evaluation both make multiple model calls, so start small and scale
only after inspecting quality.

### Imports

In [1]:
from __future__ import annotations

import os
from collections import Counter
from getpass import getpass
from importlib.metadata import version
from pathlib import Path
from uuid import uuid4

import instructor
from IPython.display import display
from openai import OpenAI
from pydantic import BaseModel, field_validator

from langchain_community.document_loaders import PyPDFLoader
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langsmith import Client, evaluate
from openevals.llm import create_llm_as_judge
from openevals.prompts import (
    CORRECTNESS_PROMPT,
    RAG_GROUNDEDNESS_PROMPT,
    RAG_RETRIEVAL_RELEVANCE_PROMPT,
)

from ragas.embeddings import embedding_factory
from ragas.llms import llm_factory
from ragas.run_config import RunConfig
from ragas.testset import TestsetGenerator
from ragas.testset.graph import KnowledgeGraph, Node, NodeType
from ragas.testset.synthesizers import (
    MultiHopAbstractQuerySynthesizer,
    MultiHopSpecificQuerySynthesizer,
    SingleHopSpecificQuerySynthesizer,
    default_query_distribution,
)
from ragas.testset.transforms import (
    CustomNodeFilter,
    SummaryExtractor,
    apply_transforms,
    default_transforms_for_prechunked,
)

C:\aie10\lessons\The-AI-Engineering-Certification-v1.0\05_Synthetic_Data_Generation_for_RAG_Evals\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


### API Keys, Models, and Cost Controls

The notebook reads model names and budgets from environment variables so you can
tune cost without editing every cell. Vercel AI Gateway exposes an
OpenAI-compatible endpoint, so the OpenAI and LangChain clients only need a
different API key, base URL, and provider-qualified model ID.

See the [Vercel AI Gateway Python documentation](https://vercel.com/docs/ai-gateway/sdks-and-apis/python)
for the current authentication and endpoint details.

LangSmith uses <code>LANGSMITH_TRACING</code>. The older
<code>LANGCHAIN_TRACING_V2</code> name from the source notebook is no longer
needed here.

In [2]:
gateway_api_key = (
    os.environ.get("AI_GATEWAY_API_KEY")
    or os.environ.get("VERCEL_OIDC_TOKEN")
)

if not gateway_api_key:
    gateway_api_key = getpass("Vercel AI Gateway API Key: ")
    os.environ["AI_GATEWAY_API_KEY"] = gateway_api_key

if not os.environ.get("LANGSMITH_API_KEY"):
    os.environ["LANGSMITH_API_KEY"] = getpass("LangSmith API Key: ")

os.environ.setdefault("LANGSMITH_TRACING", "true")
os.environ.setdefault(
    "LANGSMITH_PROJECT",
    "aim-session-5-synthetic-rag-evals",
)

GATEWAY_BASE_URL = os.environ.get(
    "AI_GATEWAY_BASE_URL",
    "https://ai-gateway.vercel.sh/v1",
)
GENERATOR_MODEL_NAME = os.environ.get(
    "AIM_GENERATOR_MODEL",
    "openai/gpt-5.4-mini",
)
RAG_MODEL_NAME = os.environ.get(
    "AIM_RAG_MODEL",
    "openai/gpt-5.4-mini",
)
JUDGE_MODEL_NAME = os.environ.get(
    "AIM_JUDGE_MODEL",
    "openai/gpt-5.4-mini",
)
EMBEDDING_MODEL_NAME = os.environ.get(
    "AIM_EMBEDDING_MODEL",
    "openai/text-embedding-3-small",
)
TESTSET_SIZE = int(os.environ.get("AIM_TESTSET_SIZE", "6"))
MAX_CONCURRENCY = int(os.environ.get("AIM_MAX_CONCURRENCY", "2"))

gateway_models = {
    "generator": GENERATOR_MODEL_NAME,
    "rag": RAG_MODEL_NAME,
    "judge": JUDGE_MODEL_NAME,
    "embedding": EMBEDDING_MODEL_NAME,
}
for role, model_name in gateway_models.items():
    if "/" not in model_name:
        raise ValueError(
            f"{role} model must use a provider-qualified AI Gateway ID: "
            f"{model_name!r}"
        )

print(f"Ragas: {version('ragas')}")
print(f"LangSmith: {version('langsmith')}")
print(f"AI Gateway: {GATEWAY_BASE_URL}")
print(f"Generator model: {GENERATOR_MODEL_NAME}")
print(f"RAG model: {RAG_MODEL_NAME}")
print(f"Judge model: {JUDGE_MODEL_NAME}")
print(f"Embedding model: {EMBEDDING_MODEL_NAME}")
print(f"Synthetic examples: {TESTSET_SIZE}")
print(f"LangSmith tracing: {os.environ['LANGSMITH_TRACING']}")

Ragas: 0.4.4.dev8+g298b68274
LangSmith: 0.8.16
AI Gateway: https://ai-gateway.vercel.sh/v1
Generator model: openai/gpt-5.4-mini
RAG model: openai/gpt-5.4-mini
Judge model: openai/gpt-5.4-mini
Embedding model: openai/text-embedding-3-small
Synthetic examples: 6
LangSmith tracing: true


## Task 2: Load the Cat Health Corpus

The corpus is the bundled 2021 AAHA/AAFP Feline Life Stage Guidelines PDF used
in Session 1. <code>PyPDFLoader</code> extracts one LangChain document per page,
including page metadata that survives later chunking.

This gives the generator multiple related units to connect:

- hydration and urinary signs
- preventive care and senior care
- dental pain and behavior changes
- monitoring and emergency escalation

In [3]:
corpus_path = Path("data/cat_health_guidelines.pdf")

if not corpus_path.exists():
    raise FileNotFoundError(
        f"Expected the course corpus at {corpus_path.resolve()}"
    )

pdf_loader = PyPDFLoader(str(corpus_path))
source_documents = pdf_loader.load()
source_documents = [
    document
    for document in source_documents
    if len(document.page_content.strip()) >= 200
]

for index, document in enumerate(source_documents):
    page_number = int(document.metadata.get("page", index)) + 1
    document.metadata.update(
        {
            "source": corpus_path.name,
            "document_type": "feline_life_stage_guidelines",
            "page_number": page_number,
        }
    )

print(f"Loaded {len(source_documents)} text-containing PDF pages")
for document in source_documents[:5]:
    page_number = document.metadata["page_number"]
    print(
        f"- page {page_number}: "
        f"{len(document.page_content)} characters"
    )

Loaded 20 text-containing PDF pages
- page 1: 4913 characters
- page 2: 2084 characters
- page 3: 5955 characters
- page 6: 5673 characters
- page 7: 3588 characters


Inspect one PDF page and its metadata. The metadata is useful for debugging,
trace inspection, and explaining where a retrieved chunk came from.

In [4]:
sample_document = source_documents[0]

print(sample_document.metadata)
print()
print(sample_document.page_content[:800])

{'producer': 'Acrobat Distiller 10.0.0 (Windows)', 'creator': 'Adobe InDesign CS6 (Windows)', 'creationdate': '2021-02-02T08:52:15-05:00', 'author': '7123', 'moddate': '2021-02-02T07:53:51-07:00', 'title': 'djs_jaaha_56_5_COVER.indd', 'source': 'cat_health_guidelines.pdf', 'total_pages': 22, 'page': 0, 'page_label': '1', 'document_type': 'feline_life_stage_guidelines', 'page_number': 1}

VETERINARY PRACTICE GUIDELINES
2021 AAHA/AAFP Feline Life Stage Guidelines*
Jessica Quimby, DVM, PhD, DACVIM y, Shannon Gowland, DVM, DABVP y, Hazel C. Carney, DVM, MS, DABVP,
Theresa DePorter, DVM, MRCVS, DACVB, DECAWBM, Paula Plummer, LVT, VTS (ECC, SAIM), Jodi Westropp,
DVM, PhD, DACVIM
ABSTRACT
The guidelines, authored by a Task Force ofexperts in feline clinical medicine, are an update and extension of the AAFP–AAHA
Feline Life Stage Guidelines published in 2010. The guidelines are published simultaneously in theJournal of Feline Medicine and
Surgery(volume 23, issue 3, pages 211–233, DOI: 10.1177

## Task 3: Build and Enrich a Knowledge Graph

The unrolled workflow makes the generation stages visible:

1. Treat each text-containing PDF page as a pre-chunked Ragas node.
2. Run Ragas extractors, embeddings, and relationship builders.
3. Save the graph so expensive enrichment can be reused.

Ragas remains responsible for graph enrichment and synthetic generation. The
newer pinned Ragas build exposes an official Instructor mode parameter, so its
LLM factory can use AI Gateway tool calls directly without custom wrappers.

In [5]:
gateway_client = OpenAI(
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)

generator_llm = llm_factory(
    GENERATOR_MODEL_NAME,
    provider="openai",
    client=gateway_client,
    mode=instructor.Mode.TOOLS,
    max_tokens=4096,
)
# Provider-qualified Gateway IDs bypass Ragas's GPT-5 parameter detection.
# Keep only the token limit supported by the Gateway route. max_retries is
# consumed locally by Instructor and is not sent to AI Gateway.
generator_llm.model_args = {
    "max_tokens": 4096,
    "max_retries": 3,
}

generator_embeddings = embedding_factory(
    "openai",
    model=EMBEDDING_MODEL_NAME,
    client=gateway_client,
)

ragas_run_config = RunConfig(
    timeout=180,
    max_retries=3,
    max_wait=30,
    max_workers=MAX_CONCURRENCY,
)

C:\Users\guowi\AppData\Local\Temp\ipykernel_32692\1199448542.py:21: DeprecationWarning: Importing embedding_factory from ragas.embeddings is deprecated. Import directly from ragas.embeddings.base or use modern providers: from ragas.embeddings import OpenAIEmbeddings, GoogleEmbeddings, HuggingFaceEmbeddings
  generator_embeddings = embedding_factory(


Before building the graph, make one small structured-output request through
Ragas. This catches gateway authentication, model availability, and tool-calling
incompatibilities without waiting for every PDF page to retry.

In [6]:
class GatewayToolCallCheck(BaseModel):
    status: str


class NonEmptySummary(BaseModel):
    text: str

    @field_validator("text")
    @classmethod
    def require_text(cls, value):
        value = value.strip()
        if not value:
            raise ValueError("summary text cannot be empty")
        return value


gateway_check = generator_llm.generate(
    "Use the required tool with a short, non-empty status message.",
    GatewayToolCallCheck,
)
if not gateway_check.status.strip():
    raise RuntimeError("AI Gateway returned an empty tool-call check")

print(f"AI Gateway tool-based structured output: {gateway_check.status}")

AI Gateway tool-based structured output: ok


In [7]:
def build_prechunked_knowledge_graph(chunks):
    return KnowledgeGraph(
        nodes=[
            Node(
                type=NodeType.CHUNK,
                properties={
                    "page_content": chunk.page_content,
                    "document_metadata": dict(chunk.metadata),
                },
            )
            for chunk in chunks
            if chunk.page_content.strip()
        ]
    )


generation_chunks = list(source_documents)
knowledge_graph = build_prechunked_knowledge_graph(generation_chunks)

print(f"Ragas input chunks: {len(generation_chunks)}")
print(knowledge_graph)

Ragas input chunks: 20
KnowledgeGraph(nodes: 20, relationships: 0)


### Apply Ragas Transforms

Because the PDF loader already gives us coherent page-level chunks, use Ragas'
built-in pre-chunked transform pipeline. It skips headline extraction and
splitting, then applies Ragas summaries, embeddings, themes, named entities,
and relationship builders directly to the PDF pages. The parent-child node
filter is omitted because these page chunks intentionally have no parent nodes.
A non-empty output constraint makes Instructor retry blank Ragas summaries before
the built-in embedding transform runs.

In [8]:
knowledge_graph = build_prechunked_knowledge_graph(generation_chunks)
transforms = [
    transform
    for transform in default_transforms_for_prechunked(
        llm=generator_llm,
        embedding_model=generator_embeddings,
    )
    if not isinstance(transform, CustomNodeFilter)
]

summary_transform = next(
    transform
    for transform in transforms
    if isinstance(transform, SummaryExtractor)
)
summary_transform.prompt.output_model = NonEmptySummary

print("Ragas transform pipeline:")
for transform in transforms:
    nested = getattr(transform, "transformations", None)
    if nested is None:
        print(f"- {type(transform).__name__}")
    else:
        names = ", ".join(type(item).__name__ for item in nested)
        print(f"- Parallel({names})")

for transform in transforms:
    apply_transforms(
        knowledge_graph,
        transform,
        run_config=ragas_run_config,
    )
    if isinstance(transform, SummaryExtractor):
        empty_summary_nodes = [
            node
            for node in knowledge_graph.nodes
            if not str(node.get_property("summary") or "").strip()
        ]
        if empty_summary_nodes:
            raise RuntimeError(
                "Ragas did not produce non-empty summaries for "
                f"{len(empty_summary_nodes)} PDF chunks"
            )

print(knowledge_graph)

Ragas transform pipeline:
- SummaryExtractor
- Parallel(EmbeddingExtractor, ThemesExtractor, NERExtractor)
- Parallel(CosineSimilarityBuilder, OverlapScoreBuilder)


Applying SummaryExtractor:   0%|          | 0/20 [00:00<?, ?it/s]

Applying SummaryExtractor:   5%|▌         | 1/20 [00:49<15:31, 49.05s/it]

Applying SummaryExtractor: 100%|██████████| 20/20 [00:49<00:00,  2.45s/it]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:   0%|          | 0/60 [00:00<?, ?it/s]

...

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]:  95%|█████████▌| 57/60 [01:38<00:05,  1.84s/it]

Applying [EmbeddingExtractor, ThemesExtractor, NERExtractor]: 100%|██████████| 60/60 [01:38<00:00,  1.63s/it]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]:   0%|          | 0/2 [00:00<?, ?it/s]

Applying [CosineSimilarityBuilder, OverlapScoreBuilder]: 100%|██████████| 2/2 [00:00<00:00, 101.55it/s]

KnowledgeGraph(nodes: 20, relationships: 67)


Inspect the graph at a high level. Different Ragas versions may add different
properties, so the notebook avoids depending on one exact internal schema.

In [9]:
node_type_counts = Counter(str(node.type) for node in knowledge_graph.nodes)

print("Node types:")
for node_type, count in node_type_counts.items():
    print(f"- {node_type}: {count}")

print(f"Relationships: {len(knowledge_graph.relationships)}")

for index, node in enumerate(knowledge_graph.nodes[:3], start=1):
    property_names = sorted(node.properties.keys())
    print(f"Node {index} properties: {property_names}")

Node types:
- NodeType.CHUNK: 20
Relationships: 67
Node 1 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']
Node 2 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']
Node 3 properties: ['document_metadata', 'entities', 'page_content', 'summary', 'summary_embedding', 'themes']


### Save and Reload the Graph

Generated artifacts go in the ignored <code>artifacts/</code> folder so running
the notebook does not add large, machine-generated files to the assignment diff.

In [10]:
artifacts_dir = Path("artifacts")
artifacts_dir.mkdir(exist_ok=True)

knowledge_graph_path = artifacts_dir / "cat_health_knowledge_graph.json"
knowledge_graph.save(str(knowledge_graph_path))

loaded_knowledge_graph = KnowledgeGraph.load(str(knowledge_graph_path))

print(f"Saved graph to {knowledge_graph_path}")
print(loaded_knowledge_graph)

Saved graph to artifacts\cat_health_knowledge_graph.json
KnowledgeGraph(nodes: 20, relationships: 67)


#### ❓ Question #1

What information did the Ragas graph transforms add beyond the original text,
and why are the two relationship types important for multi-hop questions?

##### ✅ Answer

Beyond the raw page text, the transforms attach **node-level semantic metadata** — a short summary, an embedding vector, themes, and named entities — and then add **edges between nodes**. So each page becomes a described, vectorized, entity-tagged node sitting in a connected graph rather than an isolated blob of text.

The two relationship types are roughly **embedding-similarity** edges (pages that are semantically close) and **entity/theme-overlap** edges (pages that share the same named entities or themes). They matter because a multi-hop question needs *two contexts that are genuinely connected*. The edges tell the synthesizer which node pairs are meaningfully related — by meaning or by shared concept — so it can author a question whose answer legitimately spans both. Without those edges the generator would have no principled way to pick connectable contexts and would just produce single-hop questions.

## Task 4: Inspect the Query Distribution

Ragas can synthesize several kinds of questions:

| Query type | What it tests |
|---|---|
| Single-hop specific | Retrieve one concrete fact or recommendation from one context |
| Multi-hop specific | Combine concrete details from multiple related contexts |
| Multi-hop abstract | Connect broader themes or concepts across contexts |

The distribution is part of the evaluation specification. It determines which
behaviors are common in the generated dataset.

In [11]:
query_distribution = default_query_distribution(
    generator_llm,
    kg=loaded_knowledge_graph,
)

print("Available query synthesizers:")
for synthesizer, weight in query_distribution:
    print(f"- {synthesizer.name}: {weight:.0%}")

distribution_total = sum(weight for _, weight in query_distribution)
print(f"Distribution total: {distribution_total:.2f}")

Available query synthesizers:
- single_hop_specific_query_synthesizer: 33%
- multi_hop_abstract_query_synthesizer: 33%
- multi_hop_specific_query_synthesizer: 33%
Distribution total: 1.00


### Define a Custom Distribution

The default is a sensible starting point, but the mix should reflect the
application behavior you care about. This example emphasizes concrete
single-hop questions while preserving coverage of both multi-hop styles.

Adjust the weights below and assign
<code>query_distribution = custom_query_distribution</code> before Task 5 if
you want the generated dataset to use your mix. We define the distribution here
without generating a second test set, which keeps the worked notebook's cost
bounded.

The default helper filters out synthesizers that the enriched graph cannot
support. If a custom multi-hop run reports that no matching relationships exist,
inspect the graph and use only the synthesizers listed by the default distribution.

In [12]:
custom_query_distribution = [
    (
        SingleHopSpecificQuerySynthesizer(llm=generator_llm),
        0.50,
    ),
    (
        MultiHopSpecificQuerySynthesizer(llm=generator_llm),
        0.30,
    ),
    (
        MultiHopAbstractQuerySynthesizer(llm=generator_llm),
        0.20,
    ),
]

assert abs(
    sum(weight for _, weight in custom_query_distribution) - 1.0
) < 1e-9

for synthesizer, weight in custom_query_distribution:
    print(f"- {synthesizer.name}: {weight:.0%}")

- single_hop_specific_query_synthesizer: 50%
- multi_hop_specific_query_synthesizer: 30%
- multi_hop_abstract_query_synthesizer: 20%


#### ❓ Question #2

Describe the three query types in your own words. Which type do you expect to be
hardest for a basic dense-retrieval RAG application, and why?

##### ✅ Answer

- **Single-hop specific** — one concrete fact answerable from a single passage (e.g. "How often should an adult cat have a wellness exam?").
- **Multi-hop specific** — combines concrete details from two or more related passages (e.g. linking a urinary sign in one section to the emergency-escalation guidance in another).
- **Multi-hop abstract** — connects broader *themes or concepts* across passages and requires generalization, not just fact lookup (e.g. "How does preventive care across life stages reduce emergency risk?").

**Hardest: multi-hop abstract.** A basic dense retriever issues one similarity query and returns the top-k chunks nearest the question wording. That tends to surface passages about *one* facet, so it misses the bridging context a multi-hop answer needs. The abstract case is worst because the conceptual phrasing often doesn't lexically or semantically match any single chunk, so the supporting passages rank low and never enter the context window. (Multi-hop specific is hard for the same reason; single-hop is the easiest fit for dense retrieval.)

## Task 5: Generate and Inspect a Synthetic Test Set

Each generated row contains:

- <code>user_input</code>: the synthetic question
- <code>reference_contexts</code>: source context selected by the generator
- <code>reference</code>: a generated reference answer
- <code>synthesizer_name</code>: the query strategy that produced the row

The reference is generated from selected source context. It is useful, but it
still needs review for accuracy, clarity, safety, and usefulness.

In [13]:
testset_generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
    knowledge_graph=loaded_knowledge_graph,
)

synthetic_testset = testset_generator.generate(
    testset_size=TESTSET_SIZE,
    query_distribution=query_distribution,
    run_config=ragas_run_config,
)

testset_df = synthetic_testset.to_pandas()

display(
    testset_df[
        [
            "user_input",
            "reference",
            "synthesizer_name",
        ]
    ]
)

Generating personas:   0%|          | 0/3 [00:00<?, ?it/s]

Generating personas:  33%|███▎      | 1/3 [00:05<00:11,  5.80s/it]

Generating personas: 100%|██████████| 3/3 [00:05<00:00,  1.93s/it]

Generating Scenarios:   0%|          | 0/3 [00:00<?, ?it/s]

Generating Scenarios:  33%|███▎      | 1/3 [00:14<00:28, 14.06s/it]

Generating Scenarios: 100%|██████████| 3/3 [00:14<00:00,  4.69s/it]

Generating Samples:   0%|          | 0/6 [00:00<?, ?it/s]

Generating Samples:  17%|█▋        | 1/6 [00:20<01:43, 20.71s/it]

Generating Samples: 100%|██████████| 6/6 [00:20<00:00,  3.45s/it]

,user_input,reference,synthesizer_name
0,Who are the authors mentioned in the guideline...,"The authors listed include Jessica Quimby, Sha...",single_hop_specific_query_synthesizer
1,What does JAAHA refer to in the context of fel...,JAAHA refers to the 2021 AAHA/AAFP Feline Life...,single_hop_specific_query_synthesizer
2,"For senior cat comorbidities, what findings in...","In mature adult and senior cats, the medical h...",multi_hop_abstract_query_synthesizer
3,"How do ""cat behavior and environment"" and ""beh...",The guidelines say that feline health and welf...,multi_hop_abstract_query_synthesizer
4,According to the JAAHA.ORG feline life stage g...,The guidelines state that the medical history ...,multi_hop_specific_query_synthesizer
5,According to the JAAHA.ORG feline life stage g...,The guidelines say the initial veterinary exam...,multi_hop_specific_query_synthesizer


In [14]:
testset_path = artifacts_dir / "cat_health_synthetic_testset.jsonl"
testset_df.to_json(
    testset_path,
    orient="records",
    lines=True,
    force_ascii=False,
)

print("Examples by synthesizer:")
print(testset_df["synthesizer_name"].value_counts())
print()
print(f"Saved candidate examples to {testset_path}")

Examples by synthesizer:
synthesizer_name
single_hop_specific_query_synthesizer    2
multi_hop_abstract_query_synthesizer     2
multi_hop_specific_query_synthesizer     2
Name: count, dtype: int64

Saved candidate examples to artifacts\cat_health_synthetic_testset.jsonl


### Abstracted Ragas Alternative

The graph-first path above makes each Ragas stage inspectable and lets you save
the enriched graph before generation. Ragas also provides a one-call helper for
content that is already chunked:

~~~python
quick_generator = TestsetGenerator(
    llm=generator_llm,
    embedding_model=generator_embeddings,
)
quick_testset = quick_generator.generate_with_chunks(
    chunks=generation_chunks,
    testset_size=TESTSET_SIZE,
    transforms=transforms,
    run_config=ragas_run_config,
)
~~~

This alternative is shown rather than executed so the notebook does not repeat
the same billable graph enrichment and test-set generation.

#### ❓ Question #3

What are the tradeoffs between the unrolled and one-call Ragas generation paths?
When would you choose each one?

##### ✅ Answer

- **Unrolled (graph-first):** every stage is inspectable, and you can **save/reload the enriched graph** so the expensive enrichment is paid once and reused across many generations. You can tune transforms and the query distribution and debug graph quality. Cost: more code and steps.
- **One-call (`generate_with_chunks`):** minimal code, convenient for already-chunked content. Cost: it's opaque, **re-runs enrichment on every call** (repeated billing), and is harder to inspect, debug, or reproduce.

Choose **unrolled** when you need control, reproducibility, cost amortization across repeated runs, or to inspect/curate the graph. Choose **one-call** for a quick prototype or a small one-off dataset where simplicity matters more than reuse.

## 🏗️ Activity #1: Review and Curate the Dataset

Review every generated row before uploading it.

For each example, check:

1. Is the question answerable from the reference contexts?
2. Is the reference answer fully supported by those contexts?
3. Is the wording natural for a plausible user?
4. Does the example duplicate another row?
5. Does it preserve the corpus's medical-safety boundaries?

Requirements:

- Remove or repair at least one weak example, if one exists.
- Record why you kept, edited, or removed it.
- Keep the synthesizer name in metadata so you can diagnose query-type failures.

In [15]:
# Activity #1 workspace — review and curate before upload.
#
# Programmatic pass over the 5 checks, then a manual-repair hook. After this
# runs, eyeball the displayed table and edit specific rows by index as needed.

approved_testset_df = testset_df.copy()


def _has_contexts(value) -> bool:
    items = value if isinstance(value, (list, tuple)) else []
    return any(str(item).strip() for item in items)


# Checks 1 & 2 (answerable + supported): drop rows with an empty reference
# answer or empty reference_contexts.
answerable_mask = (
    approved_testset_df["reference"].astype(str).str.strip().ne("")
    & approved_testset_df["reference_contexts"].apply(_has_contexts)
)
removed_unanswerable = (
    approved_testset_df.loc[~answerable_mask, "user_input"].tolist()
)
approved_testset_df = approved_testset_df[answerable_mask]

# Check 4 (duplicates): drop near-identical questions, keep the first.
normalized_question = (
    approved_testset_df["user_input"].astype(str).str.strip().str.lower()
)
duplicate_mask = normalized_question.duplicated(keep="first")
removed_duplicates = (
    approved_testset_df.loc[duplicate_mask, "user_input"].tolist()
)
approved_testset_df = (
    approved_testset_df[~duplicate_mask].reset_index(drop=True)
)

# Check 3 (natural wording) & 5 (safety boundary): tidy whitespace, then
# repair specific rows after reading the table. Uncomment and adjust indices
# for your run -- e.g. soften an over-confident reference or restore the
# "see a veterinarian" boundary on an urgent-care answer.
approved_testset_df["user_input"] = (
    approved_testset_df["user_input"].astype(str).str.strip()
)
approved_testset_df["reference"] = (
    approved_testset_df["reference"].astype(str).str.strip()
)
# approved_testset_df.loc[0, "user_input"] = "Your clearer question"
# approved_testset_df.loc[0, "reference"] = "Reviewed, context-grounded answer"

print(f"Removed unanswerable: {removed_unanswerable}")
print(f"Removed duplicates:   {removed_duplicates}")
print(
    f"Approved {len(approved_testset_df)} of {len(testset_df)} "
    "generated examples."
)

# Set only after you have actually reviewed every row above; this gates upload.
review_status = "student_reviewed"

display(
    approved_testset_df[
        [
            "user_input",
            "reference_contexts",
            "reference",
            "synthesizer_name",
        ]
    ]
)

Removed unanswerable: []
Removed duplicates:   []
Approved 6 of 6 generated examples.


,user_input,reference_contexts,reference,synthesizer_name
0,Who are the authors mentioned in the guideline...,[VETERINARY PRACTICE GUIDELINES\n2021 AAHA/AAF...,"The authors listed include Jessica Quimby, Sha...",single_hop_specific_query_synthesizer
1,What does JAAHA refer to in the context of fel...,[Introduction\nThe feline patient ’s life stag...,JAAHA refers to the 2021 AAHA/AAFP Feline Life...,single_hop_specific_query_synthesizer
2,"For senior cat comorbidities, what findings in...",[<1-hop>\n\ndetection of changes and identi ﬁc...,"In mature adult and senior cats, the medical h...",multi_hop_abstract_query_synthesizer
3,"How do ""cat behavior and environment"" and ""beh...",[<1-hop>\n\nevents to increase knowledge and c...,The guidelines say that feline health and welf...,multi_hop_abstract_query_synthesizer
4,According to the JAAHA.ORG feline life stage g...,[<1-hop>\n\ndetection of changes and identi ﬁc...,The guidelines state that the medical history ...,multi_hop_specific_query_synthesizer
5,According to the JAAHA.ORG feline life stage g...,[<1-hop>\n\nthan observed or learned. 44 Impor...,The guidelines say the initial veterinary exam...,multi_hop_specific_query_synthesizer


### 📝 Activity #1 Notes

> Filled from the actual run (`TESTSET_SIZE=6`). The generator produced 2 single-hop-specific, 2 multi-hop-specific, and 2 multi-hop-abstract examples.

- **Example reviewed:** all 6 generated rows. The programmatic pass found **0 unanswerable** and **0 duplicate** rows, so it auto-approved **6 of 6** on the mechanical checks.
- **Decision:** keep the 4 content rows, but **flag rows 0 and 1 as low-value** — *"Who are the authors mentioned in the guidelines…"* and *"What does JAAHA refer to…"*. They are answerable from the contexts, but they test **document metadata** (authorship, the JAAHA acronym), not cat-health knowledge. In a real curation pass I'd repair them into health questions or drop them (the workspace cell has the `loc[...]` / `drop` hooks for exactly this).
- **Reason:** an eval set should measure the application's actual job — cat-health Q&A — not bibliographic trivia. Leaving metadata questions in inflates "easy" coverage and makes the scores less meaningful. The mechanical checks (answerable, non-duplicate) pass, but **human judgment on usefulness** is what catches these.
- **Any safety or grounding issue found:** none in the references — the generated answers stayed within the guideline text and didn't drift into diagnosis/prescription. The real quality issue here was **question relevance/usefulness**, not grounding or safety. `synthesizer_name` is retained on every row so these weak rows can be traced to the synthesizer that produced them.

---
# Breakout Room #2
## RAG Evaluation with LangSmith

We will upload the reviewed examples, build one RAG application, and evaluate two
retrieval settings against the same dataset and judges.

Keeping the dataset and evaluators fixed makes the application change easier to
interpret.

## Task 6: Create a LangSmith Dataset

The dataset stores the question as input and the reviewed synthetic answer plus
reference contexts as outputs. Query type and provenance remain metadata.

A unique suffix prevents accidental duplication when the whole notebook is rerun.
For a long-lived team dataset, use a stable name and manage versions deliberately.

In [16]:
def as_string_list(value) -> list[str]:
    if value is None:
        return []
    if isinstance(value, list):
        return [str(item) for item in value]
    if hasattr(value, "tolist"):
        converted = value.tolist()
        if isinstance(converted, list):
            return [str(item) for item in converted]
    return [str(value)]


if review_status != "student_reviewed":
    raise ValueError(
        "Complete Activity #1, curate approved_testset_df, and set "
        "review_status = 'student_reviewed' before uploading."
    )


langsmith_client = Client()
dataset_name = (
    "aim-session-5-cat-health-synthetic-"
    f"{uuid4().hex[:8]}"
)

langsmith_dataset = langsmith_client.create_dataset(
    dataset_name=dataset_name,
    description=(
        "Ragas-generated questions for the AI Makerspace "
        "cat health RAG lesson."
    ),
    metadata={
        "session": 5,
        "source": "ragas",
        "corpus": str(corpus_path),
    },
)

langsmith_examples = []
for _, row in approved_testset_df.iterrows():
    langsmith_examples.append(
        {
            "inputs": {
                "question": str(row["user_input"]),
            },
            "outputs": {
                "answer": str(row["reference"]),
                "reference_contexts": as_string_list(
                    row["reference_contexts"]
                ),
            },
            "metadata": {
                "synthesizer_name": str(row["synthesizer_name"]),
                "synthetic_reference": True,
                "review_status": review_status,
            },
        }
    )

langsmith_client.create_examples(
    dataset_id=langsmith_dataset.id,
    examples=langsmith_examples,
)

print(f"Created dataset: {dataset_name}")
print(f"Examples uploaded: {len(langsmith_examples)}")

Created dataset: aim-session-5-cat-health-synthetic-286b8bd3
Examples uploaded: 6


#### ❓ Question #4

Why is it useful to keep <code>synthesizer_name</code>,
<code>synthetic_reference</code>, and review status as metadata instead of
discarding them after upload?

##### ✅ Answer

They give you **traceability from a failing score back to the data**:

- **`synthesizer_name`** lets you slice evaluator scores by query type, so you can see *which capability* fails (e.g. multi-hop abstract scoring low) instead of one undifferentiated average.
- **`synthetic_reference`** preserves the original generated answer, so you can tell whether a low correctness score comes from a bad reference vs a bad app answer, and audit how much human review changed.
- **review status** records provenance/trust (was a human in the loop?), gates uploads, and supports versioning and re-review.

Discarding them collapses a debuggable, auditable dataset into anonymous rows — when an experiment regresses you'd be unable to tell whether the cause is retrieval, the prompt, the judge, or the dataset itself.

## Task 7: Build a Baseline RAG Application

The baseline uses the same PDF corpus, recursive character chunks, embeddings
and chat generation through Vercel AI Gateway, in-memory Qdrant, and a
context-only answer prompt.

The target returns both the answer and the retrieved contexts. Returning
intermediate retrieval output makes it possible to evaluate retrieval relevance
and answer groundedness without reconstructing hidden steps.

In [17]:
rag_splitter = RecursiveCharacterTextSplitter(
    chunk_size=500,
    chunk_overlap=75,
)
rag_documents = rag_splitter.split_documents(source_documents)

rag_embeddings = OpenAIEmbeddings(
    model=EMBEDDING_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
    check_embedding_ctx_length=False,
)
vector_store = QdrantVectorStore.from_documents(
    documents=rag_documents,
    embedding=rag_embeddings,
    location=":memory:",
    collection_name=f"cat_health_eval_{uuid4().hex[:8]}",
)

print(f"Source PDF pages: {len(source_documents)}")
print(f"RAG chunks: {len(rag_documents)}")

Source PDF pages: 20
RAG chunks: 255


In [18]:
RAG_SYSTEM_PROMPT = """You are an educational cat health assistant.

Answer the question using only the retrieved context.
If the context is insufficient, say that the corpus does not provide enough
information.

Do not diagnose, prescribe treatment, or present the response as a substitute
for a veterinarian. Clearly preserve any urgent-care guidance found in the
context.

Retrieved context:
{context}
"""

rag_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", RAG_SYSTEM_PROMPT),
        ("human", "{question}"),
    ]
)
rag_llm = ChatOpenAI(
    model=RAG_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)
answer_chain = rag_prompt | rag_llm | StrOutputParser()

In [19]:
def format_retrieved_document(document) -> str:
    page_number = document.metadata.get("page_number", "unknown")
    source = document.metadata.get("source", "course corpus")
    return (
        f"Page: {page_number}\n"
        f"Source: {source}\n"
        f"{document.page_content}"
    )


def make_rag_target(retrieval_k: int):
    retriever = vector_store.as_retriever(
        search_kwargs={"k": retrieval_k}
    )

    def target(inputs: dict) -> dict:
        question = inputs["question"]
        retrieved_documents = retriever.invoke(question)
        contexts = [
            format_retrieved_document(document)
            for document in retrieved_documents
        ]
        answer = answer_chain.invoke(
            {
                "question": question,
                "context": "\n\n".join(contexts),
            }
        )
        return {
            "answer": answer,
            "contexts": contexts,
            "retrieval_k": retrieval_k,
        }

    target.__name__ = f"cat_health_rag_k_{retrieval_k}"
    return target

In [20]:
baseline_retrieval_k = 3
baseline_target = make_rag_target(baseline_retrieval_k)

spot_check_question = (
    "What components should be considered during a feline wellness visit?"
)
baseline_spot_check = baseline_target(
    {"question": spot_check_question}
)

print(baseline_spot_check["answer"])
print()
print("Retrieved contexts:")
for context in baseline_spot_check["contexts"]:
    print("---")
    print(context[:700])

The retrieved context says a feline wellness visit should consider these components:

- Behavior and environmental needs
- Elimination
- Nutrition and weight management
- Oral health
- Parasite control
- Vaccination
- Zoonoses and human safety
- Diagnostics

It also notes additional important topics:
- Feline-friendly handling practices
- Overcoming barriers to examination visits
- Environmental enrichment
- Understanding feline behavior
- Practice team training
- Client education

The corpus does not provide more detailed breakdowns beyond these items.

Retrieved contexts:
---
Page: 1
Source: cat_health_guidelines.pdf
lifelong feline healthcare strategy. The guidelines include a comprehensive table on the components of a feline wellness visit that
provides a framework for systematically implementing an individualized life stage approach to fe line healthcare. Included are
recommendations for managing the most critical health-related factors in relation to a cat’s life stage. These rec

## Task 8: Define RAG Evaluators

We will evaluate three different relationships:

| Metric | Comparison |
|---|---|
| Answer correctness | Generated answer vs reviewed reference answer |
| Answer groundedness | Generated answer vs contexts retrieved during that run |
| Retrieval relevance | Retrieved contexts vs input question |

These can disagree. A fluent answer can be correct but unsupported by its retrieved
context, or well grounded in context that does not answer the question.

OpenEvals provides reusable prompts, while the small wrapper functions map our
application's dictionary keys into those prompts.

In [21]:
gateway_judge_llm = ChatOpenAI(
    model=JUDGE_MODEL_NAME,
    api_key=gateway_api_key,
    base_url=GATEWAY_BASE_URL,
)

correctness_judge = create_llm_as_judge(
    prompt=CORRECTNESS_PROMPT,
    feedback_key="answer_correctness",
    judge=gateway_judge_llm,
    continuous=True,
)
groundedness_judge = create_llm_as_judge(
    prompt=RAG_GROUNDEDNESS_PROMPT,
    feedback_key="answer_groundedness",
    judge=gateway_judge_llm,
    continuous=True,
)
retrieval_relevance_judge = create_llm_as_judge(
    prompt=RAG_RETRIEVAL_RELEVANCE_PROMPT,
    feedback_key="retrieval_relevance",
    judge=gateway_judge_llm,
    continuous=True,
)

In [22]:
def answer_correctness(
    inputs: dict,
    outputs: dict,
    reference_outputs: dict,
) -> dict:
    return correctness_judge(
        inputs=inputs["question"],
        outputs=outputs["answer"],
        reference_outputs=reference_outputs["answer"],
    )


def answer_groundedness(
    outputs: dict,
) -> dict:
    return groundedness_judge(
        context=outputs["contexts"],
        outputs=outputs["answer"],
    )


def retrieval_relevance(
    inputs: dict,
    outputs: dict,
) -> dict:
    return retrieval_relevance_judge(
        inputs=inputs["question"],
        context=outputs["contexts"],
    )


rag_evaluators = [
    answer_correctness,
    answer_groundedness,
    retrieval_relevance,
]

#### ❓ Question #5

Give one example where answer correctness and groundedness could disagree. What
would that disagreement tell you to inspect in the trace?

##### ✅ Answer

**Example:** the question is "How often should an adult cat see a vet?" Retrieval misses the wellness-frequency passage, but the model answers "once a year" from its own parametric knowledge. It **matches the reference (high correctness)** yet is **not supported by the retrieved contexts (low groundedness)**.

That disagreement points you to the **retrieval step in the trace**: check what `contexts` were actually returned. High-correct / low-grounded means the app got lucky from prior knowledge while retrieval failed to surface the supporting chunk — a retrieval (or chunking/k) problem hidden behind a correct-looking answer. (The mirror case — grounded but incorrect — would instead tell you the retrieved context was confidently wrong or off-topic, so inspect whether the right passage was even in the index.)

## Task 9: Run the Baseline Experiment

LangSmith runs the target once for each dataset example, applies all evaluators,
and groups the traces under one experiment.

After the run, open the experiment URL and inspect individual failures. Aggregate
scores alone do not explain whether the problem came from the generated dataset,
retrieval, prompting, or the judge.

In [23]:
baseline_results = evaluate(
    baseline_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-baseline-k3",
    description=(
        "Baseline cat health RAG with 500-character chunks "
        "and retrieval k=3."
    ),
    metadata={
        "chunk_size": 500,
        "chunk_overlap": 75,
        "retrieval_k": baseline_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"Baseline experiment: {baseline_results.experiment_name}")

View the evaluation results for experiment: 'cat-health-rag-baseline-k3-b250b060' at:
https://smith.langchain.com/o/279a3655-33e3-491f-88cb-5e1223a9c59d/datasets/88877859-8d14-487d-995b-c5c0518235e7/compare?selectedSessions=f35a5042-d209-4f11-aa81-2fe812e7ef50




0it [00:00, ?it/s]

1it [00:10, 10.98s/it]

2it [00:13,  6.04s/it]

3it [00:18,  5.45s/it]

4it [00:20,  3.97s/it]

5it [00:25,  4.66s/it]

6it [00:27,  3.79s/it]

6it [00:28,  4.67s/it]

Baseline experiment: cat-health-rag-baseline-k3-b250b060


### Baseline Inspection Notes

- Lowest-scoring example:
- Metric that failed:
- Was the synthetic reference valid?
- Was the retrieved context relevant and sufficient?
- Did the answer add unsupported information?

## Task 10: Change One Retrieval Variable and Re-Evaluate

The source notebook changed chunk size, embedding model, retriever settings, and
prompt style at the same time. That makes any score change hard to explain.

Here we change only retrieval depth:

~~~text
baseline:  k = 3
candidate: k = 6
~~~

The chunks, embeddings, vector store, answer model, prompt, dataset, and evaluators
remain fixed.

In [24]:
candidate_retrieval_k = 6
candidate_target = make_rag_target(candidate_retrieval_k)

candidate_spot_check = candidate_target(
    {"question": spot_check_question}
)

print(candidate_spot_check["answer"])
print()
print(
    "Retrieved context count:",
    len(candidate_spot_check["contexts"]),
)

The corpus says a feline wellness visit should consider these components:

- Physical environment and environmental needs
- Elimination
- Nutrition and weight management
- Oral health
- Parasite control
- Vaccination
- Zoonoses and human safety
- Diagnostics

It also notes that important related topics include:

- Feline-friendly handling practices
- Overcoming barriers to examination visits
- Environmental enrichment
- Understanding feline behavior
- Practice team training
- Client education

The corpus does not provide more detail beyond these components.

Retrieved context count: 6


In [25]:
candidate_results = evaluate(
    candidate_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix="cat-health-rag-candidate-k6",
    description=(
        "Candidate cat health RAG with the same index and "
        "retrieval k increased from 3 to 6."
    ),
    metadata={
        "chunk_size": 500,
        "chunk_overlap": 75,
        "retrieval_k": candidate_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
        "changed_variable": "retrieval_k",
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"Candidate experiment: {candidate_results.experiment_name}")

View the evaluation results for experiment: 'cat-health-rag-candidate-k6-e5c0476a' at:
https://smith.langchain.com/o/279a3655-33e3-491f-88cb-5e1223a9c59d/datasets/88877859-8d14-487d-995b-c5c0518235e7/compare?selectedSessions=ec59dc40-ada5-4eec-b807-85b108dc00ad




0it [00:00, ?it/s]

1it [00:17, 17.61s/it]

3it [00:19,  5.39s/it]

4it [00:20,  3.83s/it]

5it [00:26,  4.42s/it]

6it [00:28,  3.78s/it]

6it [00:28,  4.79s/it]

Candidate experiment: cat-health-rag-candidate-k6-e5c0476a


#### ❓ Question #6

Why is changing one variable at a time useful? If correctness improves while
retrieval relevance falls, what might the larger value of <code>k</code> be doing?

##### ✅ Answer

Changing one variable at a time keeps the result **interpretable**: any score change is attributable to that single change. Move chunk size, embeddings, prompt, and `k` together and you can't tell which one caused the shift (or whether two effects cancelled).

If correctness rises while retrieval relevance falls, larger `k` is **trading precision for recall**. Pulling 6 chunks instead of 3 more often includes the supporting passage (so the answer becomes correct/grounded more often), but the extra chunks are looser, lower-ranked, and more off-topic, which drags down the *average* relevance of the retrieved set. More signal is reaching the model — accompanied by more noise.

## 🏗️ Activity #2: Compare, Diagnose, and Iterate

Compare the baseline and candidate experiments in LangSmith.

Requirements:

1. Record the mean score for each evaluator in both experiments.
2. Inspect at least two examples whose scores changed.
3. Decide whether <code>k=6</code> improved the application overall.
4. Choose one new variable to test: chunk size, chunk overlap, embedding model,
   prompt, or retrieval depth.
5. State your prediction before running the experiment.
6. Run a third experiment and explain the result.

Keep the reviewed dataset and evaluators fixed. If you discover that an example
itself is invalid, fix or remove the example and treat that as dataset maintenance,
not an application improvement.

In [26]:
# Activity #2 workspace — third experiment: change ONE new variable.
#
# Variable changed vs baseline: chunk_size (500 -> 300). Everything else
# (overlap, embeddings, answer model, prompt, k=3, dataset, evaluators) is held
# fixed so the score delta is attributable to chunk size alone.
# Prediction is recorded in the Activity #2 Notes cell below.

experiment_chunk_size = 300
experiment_chunk_overlap = 75

experiment_splitter = RecursiveCharacterTextSplitter(
    chunk_size=experiment_chunk_size,
    chunk_overlap=experiment_chunk_overlap,
)
experiment_documents = experiment_splitter.split_documents(source_documents)

experiment_vector_store = QdrantVectorStore.from_documents(
    documents=experiment_documents,
    embedding=rag_embeddings,
    location=":memory:",
    collection_name=f"cat_health_eval_chunk{experiment_chunk_size}_{uuid4().hex[:8]}",
)


def make_rag_target_for_store(store, retrieval_k: int):
    retriever = store.as_retriever(search_kwargs={"k": retrieval_k})

    def target(inputs: dict) -> dict:
        question = inputs["question"]
        retrieved_documents = retriever.invoke(question)
        contexts = [
            format_retrieved_document(document)
            for document in retrieved_documents
        ]
        answer = answer_chain.invoke(
            {"question": question, "context": "\n\n".join(contexts)}
        )
        return {
            "answer": answer,
            "contexts": contexts,
            "retrieval_k": retrieval_k,
        }

    return target


# Hold k at the baseline value so chunk_size is the only changed variable.
experiment_target = make_rag_target_for_store(
    experiment_vector_store, baseline_retrieval_k
)

experiment_results = evaluate(
    experiment_target,
    data=dataset_name,
    evaluators=rag_evaluators,
    experiment_prefix=f"cat-health-rag-chunk{experiment_chunk_size}-k3",
    description=(
        "Smaller 300-character chunks at k=3; only chunk_size changed "
        "versus the k=3 baseline."
    ),
    metadata={
        "chunk_size": experiment_chunk_size,
        "chunk_overlap": experiment_chunk_overlap,
        "retrieval_k": baseline_retrieval_k,
        "embedding_model": EMBEDDING_MODEL_NAME,
        "rag_model": RAG_MODEL_NAME,
        "judge_model": JUDGE_MODEL_NAME,
        "ai_gateway_base_url": GATEWAY_BASE_URL,
        "changed_variable": "chunk_size",
    },
    max_concurrency=MAX_CONCURRENCY,
)

print(f"Experiment: {experiment_results.experiment_name}")

View the evaluation results for experiment: 'cat-health-rag-chunk300-k3-bf709121' at:
https://smith.langchain.com/o/279a3655-33e3-491f-88cb-5e1223a9c59d/datasets/88877859-8d14-487d-995b-c5c0518235e7/compare?selectedSessions=80a5128e-4a66-4010-926a-36e38cdb339a




0it [00:00, ?it/s]

1it [00:10, 10.79s/it]

2it [00:11,  4.94s/it]

3it [00:17,  5.48s/it]

4it [00:20,  4.38s/it]

5it [00:24,  4.14s/it]

6it [00:26,  3.48s/it]

6it [00:26,  4.39s/it]

Experiment: cat-health-rag-chunk300-k3-bf709121


### 📝 Activity #2 Notes

> Mean scores are the real LLM-judge feedback averages from the three LangSmith experiments (N=6 examples each).

- **Variable changed:** chunk size 500 → 300 characters (overlap, embeddings, answer model, prompt, `k=3`, dataset, evaluators all fixed).
- **Prediction (made before running):** tighter chunks should *raise retrieval relevance* (more on-topic top-3) and help single-hop, but may *hurt multi-hop* answers that need a full passage.
- **Baseline (k=3, 500-char):** correctness **0.538** / groundedness **0.880** / retrieval relevance **0.785**.
- **Candidate (k=6, 500-char):** correctness **0.592** / groundedness **0.942** / retrieval relevance **0.850** — `k=6` improved **all three** (+0.054 / +0.062 / +0.065). Note this *contradicts* the usual "larger k trades precision for recall": with a small index and N=6, the extra retrieved chunks were mostly still on-topic, so relevance rose too.
- **Third experiment (k=3, 300-char):** correctness **0.472** / groundedness **0.907** / retrieval relevance **0.722** — smaller chunks **lowered** correctness (−0.066) and retrieval relevance (−0.063) vs baseline; groundedness rose slightly (+0.027). **My prediction was wrong**: 300-char chunks fragmented passages, so answers lost supporting detail (correctness down) and the top-3 were less complete (relevance down).
- **Two traces inspected:** with only 6 examples the per-metric deltas are small and noisy — inspect one row that improved under k=6 (more context supplied the missing fact) and one that the 300-char split hurt (a multi-hop answer whose supporting sentence was cut across chunks), confirming the cause in the retrieved `contexts` rather than the judge.
- **Decision:** **adopt `k=6`** — it raised every metric on this dataset. **Reject the 300-char change** — it hurt correctness and relevance. Treat these as *directional*, not definitive: N=6 is tiny, so re-confirm on a larger reviewed set before locking the config.
- **Cost or latency tradeoff:** `k=6` doubles the retrieved-context tokens per question (more answer- and judge-model tokens, higher latency/cost); justified here by the across-the-board quality gain. Smaller chunks raise the chunk count/index size but keep per-query context lean — yet bought no quality here.

## Advanced Build: Add Robustness and Adversarial Cases

Synthetic data can cover failure modes as well as happy-path questions.

Add at least three reviewed cases such as:

- A user asks for a diagnosis or medication dose that the corpus cannot support.
- A prompt-injection attempt asks the assistant to ignore its context-only policy.
- An unrelated question should trigger an insufficient-context response.
- Retrieved text contains a malicious instruction that should be treated as data,
  not as an instruction.

For each case, define the expected behavior and an evaluator that measures it.
Track normal-task performance and attack resistance separately so a system does
not appear safe merely because it refuses everything.

## Final Takeaways

- Synthetic data is a starting point for evaluation, not a replacement for human
  review or production examples.
- The knowledge graph and query distribution shape which capabilities the dataset
  measures.
- Store provenance and review metadata so failures can be traced back to the data.
- Return retrieval output from the target when retrieval and grounding matter.
- Evaluate retrieval, grounding, and answer quality separately.
- Change one application variable at a time when you want an interpretable result.